# Resultados: world models de conducción sobre commaVQ
La tabla **modelo → performance**: una fila por modelo y solo con datos actuales, leída de `results/`. Si un resultado todavía no existe, la
celda dice *pendiente* (nunca muestra un número viejo). Para producirlo: `python -m src.cli <modelo> evaluar` o la cola (`python -m src.cli cola correr`);
cada corrida queda en `results/bitacora.jsonl` con su comando exacto. El informe escrito está en `Docs/resultados.md`.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

## 1. Estado de cada modelo
Qué se entrenó, evaluó y midió. Una fila por modelo de la cola; *pendiente* hasta que su resultado exista.

In [ ]:
nbui.estado_modelos();

## 2. Tabla de video
CE, LPIPS y FVD a 1 s con IC por segmento (se remuestrean segmentos, no clips: los 4 clips de un segmento comparten ruta), FVD a 3 s, latencia del modelo (p50, p99) y p99 de la cadena. El **piso de FVD** (real contra real) es el valor que ningún modelo puede bajar; hay que leer la tabla contra él. Los latentes no entran acá: tienen su tabla propia.

In [ ]:
nbui.tabla_video();

## 3. Pareto calidad–latencia
FVD a 1 s contra ms por frame, solo entre modelos generativos y con datos actuales; la línea punteada vertical son los 50 ms de la cámara y la horizontal, el piso de FVD.

In [ ]:
nbui.pareto();

## 4. Calidad según el horizonte
LPIPS contra el video real en cada frame generado (3 s).

In [ ]:
nbui.horizonte();

## 5. Muestras
Los últimos videos que generó `demo` (real a la izquierda, predicho a la derecha). Se generan desde `notebooks/demo_clase.ipynb` o con `python -m src.cli <modelo> demo`.

In [ ]:
nbui.muestras(3);

## 6. Tabla latente
Los modelos latentes (LeWM, V-JEPA 2, V-JEPA 2-AC) no generan video: se miden por error latente contra *copiar el último latente*, probes contra el encoder sin entrenar y contra píxeles, sorpresa y plan con CEM. Cada columna lleva su referencia al lado.

In [ ]:
nbui.tabla_latente();

## 7. Tabla de fidelidad
Para cada modelo, cada decisión con su fuente y su categoría: *fiel*, *datos* (lo fuerza el dato de comma), *presupuesto* (lo fuerza el cómputo) o *propio*.

In [ ]:
from src import recipes
nbui.tabla_fidelidad(recipes.fidelidad_kinds());

## 8. Corridas, cola y bitácora

In [ ]:
nbui.estado_corridas()
nbui.panel_cola()
nbui.panel_bitacora(10);

## 9. Regenerar las tablas y figuras
`python -m src.cli reporte` arma `results/configs.json` y las figuras con lo que hay en `results/`. Con `DRY_RUN = False` lo corre de verdad (es CPU y queda en la bitácora).

In [ ]:
DRY_RUN = True
nbui.general("reporte", dry_run=DRY_RUN);